### Import and configs

In [2]:
import json
import glob
import pandas as pd
from pathlib import Path

### Clean and format category name

In [20]:
df['category'] = df['category'].apply(lambda x: x.replace('_', ' ').title())
df.head()

,asin,title,rating,review_count,sponsored,is_prime,search_position,recent_sales,price_primary,price_rrp,category,page
0,B09NXS395V,24K Gold Under Eye Patches - 60 Pack for Puffy...,4.1,14615.0,False,False,1,10K+ bought in past month,9.39,9.99,Beauty Personal Care,1
1,B091NJQ29P,Good Molecules Yerba Mate Wake Up Eye Gel - Hy...,4.2,28653.0,False,False,2,60K+ bought in past month,5.97,NaN,Beauty Personal Care,1
2,B08Z5X1L1S,Gifts for Women Gift Basket for Women - 10 Pc ...,4.6,2369.0,False,False,3,5K+ bought in past month,34.98,65.00,Beauty Personal Care,1
3,B08KT2Z93D,"eos Shea Better Body Lotion Vanilla Cashmere, ...",4.7,65741.0,False,False,4,100K+ bought in past month,9.97,10.99,Beauty Personal Care,1
4,B0CMV2DCXT,CHMI Under Eye Patches (50 Pairs) - 24K Gold E...,4.5,1021.0,False,False,5,5K+ bought in past month,9.99,NaN,Beauty Personal Care,1


### Function to parse recent sales to a number

In [21]:
def parse_recent_sales(value):
    # get the integer value from recent sales
    if not isinstance(value, str):
        return None
    
    value = value.upper()

    if 'K+' in value:
        return int(float(value.split('K+')[0].strip()) *1000)
    elif '+' in value:
        return int(value.split('+')[0].strip())
    else:
        return None
        

### Applied function to cleanup

In [23]:
df['recent_sales_approx'] = df['recent_sales'].apply(parse_recent_sales)
df.head()

,asin,title,rating,review_count,sponsored,is_prime,search_position,recent_sales,price_primary,price_rrp,category,page,recent_sales_approx
0,B09NXS395V,24K Gold Under Eye Patches - 60 Pack for Puffy...,4.1,14615.0,False,False,1,10K+ bought in past month,9.39,9.99,Beauty Personal Care,1,10000.0
1,B091NJQ29P,Good Molecules Yerba Mate Wake Up Eye Gel - Hy...,4.2,28653.0,False,False,2,60K+ bought in past month,5.97,NaN,Beauty Personal Care,1,60000.0
2,B08Z5X1L1S,Gifts for Women Gift Basket for Women - 10 Pc ...,4.6,2369.0,False,False,3,5K+ bought in past month,34.98,65.00,Beauty Personal Care,1,5000.0
3,B08KT2Z93D,"eos Shea Better Body Lotion Vanilla Cashmere, ...",4.7,65741.0,False,False,4,100K+ bought in past month,9.97,10.99,Beauty Personal Care,1,100000.0
4,B0CMV2DCXT,CHMI Under Eye Patches (50 Pairs) - 24K Gold E...,4.5,1021.0,False,False,5,5K+ bought in past month,9.99,NaN,Beauty Personal Care,1,5000.0


### Convert to integers 

In [24]:
df['review_count']         = df['review_count'].astype('Int64')       # nullable int
df['recent_sales_approx']  = df['recent_sales_approx'].astype('Int64')
df.head(10)

,asin,title,rating,review_count,sponsored,is_prime,search_position,recent_sales,price_primary,price_rrp,category,page,recent_sales_approx
0,B09NXS395V,24K Gold Under Eye Patches - 60 Pack for Puffy...,4.1,14615,False,False,1,10K+ bought in past month,9.39,9.99,Beauty Personal Care,1,10000
1,B091NJQ29P,Good Molecules Yerba Mate Wake Up Eye Gel - Hy...,4.2,28653,False,False,2,60K+ bought in past month,5.97,NaN,Beauty Personal Care,1,60000
2,B08Z5X1L1S,Gifts for Women Gift Basket for Women - 10 Pc ...,4.6,2369,False,False,3,5K+ bought in past month,34.98,65.00,Beauty Personal Care,1,5000
3,B08KT2Z93D,"eos Shea Better Body Lotion Vanilla Cashmere, ...",4.7,65741,False,False,4,100K+ bought in past month,9.97,10.99,Beauty Personal Care,1,100000
4,B0CMV2DCXT,CHMI Under Eye Patches (50 Pairs) - 24K Gold E...,4.5,1021,False,False,5,5K+ bought in past month,9.99,NaN,Beauty Personal Care,1,5000
5,B0CTK5ZTNK,"Under Eye Patches, 40 Pairs Eye Mask for Dark ...",4.5,5192,False,False,6,10K+ bought in past month,9.99,12.99,Beauty Personal Care,1,10000
6,B0CNV5SG4S,"Sleeping lip mask, Nourish & Hydrate Lip Mask ...",4.6,969,False,False,7,300+ bought in past month,4.99,6.99,Beauty Personal Care,1,300
7,B0C7W6GW8P,"e.l.f. Squeeze Me Lip Balm, Moisturizing Lip B...",4.5,19260,False,False,8,10K+ bought in past month,5.00,NaN,Beauty Personal Care,1,10000
8,B0CPCYP629,eos 24H Moisture Travel Body Lotion- Vanilla C...,4.8,3099,False,False,9,20K+ bought in past month,3.99,NaN,Beauty Personal Care,1,20000
9,B0DMTDN158,The Ordinary Glycolic Acid 7% Exfoliating Tone...,4.7,44753,False,False,10,40K+ bought in past month,7.65,9.00,Beauty Personal Care,1,40000


### New columns with the current data

In [ ]:
# Discount percentage, how much off vs original price
df['discount_pct'] = ((df['price_rrp'] - df['price_primary']) / df['price_rrp'] * 100).round(1)

# Engagement score, sales weighted by rating (higher = popular AND well rated)
df['engagement_score'] = (df['recent_sales_approx'] * df['rating']).round(0)

# Has discount flag
df['has_discount'] = df['price_rrp'].notna()

df.head(5)

,asin,title,rating,review_count,sponsored,is_prime,search_position,recent_sales,price_primary,price_rrp,category,page,recent_sales_approx,discount_pct,engagement_score,has_discount
0,B09NXS395V,24K Gold Under Eye Patches - 60 Pack for Puffy...,4.1,14615,False,False,1,10K+ bought in past month,9.39,9.99,Beauty Personal Care,1,10000,6.0,41000.0,True
1,B091NJQ29P,Good Molecules Yerba Mate Wake Up Eye Gel - Hy...,4.2,28653,False,False,2,60K+ bought in past month,5.97,NaN,Beauty Personal Care,1,60000,NaN,252000.0,False
2,B08Z5X1L1S,Gifts for Women Gift Basket for Women - 10 Pc ...,4.6,2369,False,False,3,5K+ bought in past month,34.98,65.00,Beauty Personal Care,1,5000,46.2,23000.0,True
3,B08KT2Z93D,"eos Shea Better Body Lotion Vanilla Cashmere, ...",4.7,65741,False,False,4,100K+ bought in past month,9.97,10.99,Beauty Personal Care,1,100000,9.3,470000.0,True
4,B0CMV2DCXT,CHMI Under Eye Patches (50 Pairs) - 24K Gold E...,4.5,1021,False,False,5,5K+ bought in past month,9.99,NaN,Beauty Personal Care,1,5000,NaN,22500.0,False
5,B0CTK5ZTNK,"Under Eye Patches, 40 Pairs Eye Mask for Dark ...",4.5,5192,False,False,6,10K+ bought in past month,9.99,12.99,Beauty Personal Care,1,10000,23.1,45000.0,True
6,B0CNV5SG4S,"Sleeping lip mask, Nourish & Hydrate Lip Mask ...",4.6,969,False,False,7,300+ bought in past month,4.99,6.99,Beauty Personal Care,1,300,28.6,1380.0,True
7,B0C7W6GW8P,"e.l.f. Squeeze Me Lip Balm, Moisturizing Lip B...",4.5,19260,False,False,8,10K+ bought in past month,5.00,NaN,Beauty Personal Care,1,10000,NaN,45000.0,False
8,B0CPCYP629,eos 24H Moisture Travel Body Lotion- Vanilla C...,4.8,3099,False,False,9,20K+ bought in past month,3.99,NaN,Beauty Personal Care,1,20000,NaN,96000.0,False
9,B0DMTDN158,The Ordinary Glycolic Acid 7% Exfoliating Tone...,4.7,44753,False,False,10,40K+ bought in past month,7.65,9.00,Beauty Personal Care,1,40000,15.0,188000.0,True


## ⚠️ Data Limitations
- `recent_sales_approx` is an estimate derived from Amazon's "X+ bought in past month" 
  string, which only provides a lower bound (e.g. "10K+" could mean 10K or 500K).
- `engagement_score` should be interpreted as a relative ranking metric, not an absolute value.

### Save products df in a JSON file

In [26]:
df.to_csv('../data/processed/products_search.csv', index=False)
print(f"Saved: {df.shape}")


Saved: (1386, 16)
